# పాఠం 04 - సాధనం ఉపయోగం డిజైన్ ప్యాటర్న్

ఈ పాఠంలో మీరు Microsoft Agent Framework (Python) ఉపయోగించి AI ఏజెంట్ల కోసం **సాధనం ఉపయోగం** డిజైన్ ప్యాటర్న్ నేర్చుకుంటారు. ఇందులో చర్చించబడింది:

- `@tool` డెకరేటర్ మరియు టైప్ చేసిన పారామితులతో ఫంక్షన్ టూల్స్ నిర్వచించడం
- మోడల్ కి ఏ టూల్ ఏం చేయాలో తెలుసుకోవడానికి టూల్ స్కీమాలు అందించడం
- `approval_mode` తో టూల్ నిర్వహణను నియంత్రించడం
- Pydantic మోడల్స్ మరియు `response_format` ద్వారా **ఫార్మాట్ చేసిన అవుట్‌పుట్** ను కలుపుట

సన్నివేశం ఒక **ప్రయాణ బుకింగ్ ఏజెంట్** ఇది గమ్యస్థానాలను చూడడం, అందుబాటును తనిఖీ చేయడం, మరియు విమాన సమాచారం పొందగలదు.


## సెటప్


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## @tool డికొరేటర్‌తో టూల్స్‌ను నిర్వచించడం

`@tool` డికొరేటర్ ఒక సరళమైన Python ఫంక్షన్‌ను ఏజెంట్ కాల్ చేయగల టూల్‌గా మార్చుతుంది.
ముఖ్యమైన అంశాలు:

- **docstring** మోడల్ చూస్తున్న టూల్ వివరణ అవుతుంది.
- **టైప్ యానోటేషన్లు** (`Annotated` వివరణలతో సహా) టూల్ స్కీమాను నిర్వచిస్తాయి.
- `approval_mode` ప్రతి కాల్ 실행కు ముందు వినియోగదారుడు ఆమోదించాల్సిన అవసరమని నియంత్రిస్తుంది.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## బహుళ పరికరాలతో ఏజెంట్ సృష్టించటం

వినియోగదారుడి ప్రశ్నకు సమాధానం చెప్పేందుకు ఏ పరికరాన్ని అవసరం అనుకుంటే ఆ పరికరాన్ని మోడల్ పిలవగలుగుదల కోసం ముగ్గురు పరికరాలను క్లయింట్‌కు అందించండి.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = client.as_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## టూల్స్ తో నిర్మిత అవుట్పుట్

`response_format`ని Pydantic మోడల్‌గా సెట్ చేయడం ద్వారా, ఏజెంట్ స్వేచ్ఛాయుత పాఠ్యం స్థానంలో బాగా టైప్డ్ JSON ఆబ్జెక్ట్‌ను తిరిగి ఇవ్వడానికి నిర్బంధితం అవుతాడు. ఇది దిగువ కోడ్ ఫలితాన్ని ప్రోగ్రామాటిక్గా వినియోగించుకునే అవసరం ఉన్నప్పుడు ఉపయోగకరంగా ఉంటుంది.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = client.as_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## టూల్ అంగీకార నమూనాలు

`@tool` పై `approval_mode` పరామితి టూల్ కాల్స్ అమలు చేసేముందు మానవ అంగీకారం అవసరమో కాదో నియంత్రిస్తుంది:

| మోడ్ | ప్రవర్తన |
|---|---|
| `"never_require"` | టూల్ ఆటోమేటిక్‌గా నడుస్తుంది — వినియోగదారు నిర్ధారణ అవసరం లేదు. |
| `"always_require"` | ప్రతి కాల్ అమలు చేసే ముందు వినియోగదారు ఆమోదం తప్పనిసరి. |

పక్కప్రభావాలు ఉన్న టూల్స్ కోసం `"always_require"` ఉపయోగించండి (ఉదా: విమానం బుక్ చేయడం, క్రెడిట్ కార్డు ఛార్జ్ చేయడం) ఇది మానవుడిని చర్యలో ఉంచుతుంది.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## సంగ్రహం

ఈ పాఠంలో మీరు నేర్చుకున్నది:

1. టూల్ స్కీమా గా పనిచేసే టైప్ చేయబడిన పారామితులు మరియు డాక్‌స్ట్రింగ్లతో `@tool` డెకరేటర్ ఉపయోగించి **టూల్స్‌ను నిర్వచించడం**.
2. ఏజెంట్ క్లిష్టమైన ప్రశ్నలకు సమాధానం చెప్పడానికి క్రమంగా పిలవగలిగే విధంగా **బహుళ టూల్స్‌ను రూపొందించడం**.
3. `response_format`గా Pydantic మోడల్ ను పాస్ చేస్తూ **సంఘటిత అవుట్పుట్ ను అందించడం**.
4. సున్నితమైన ఆపరేషన్ల కోసం మానవ నియంత్రణ కోసం `approval_mode` తో **టూల్ ఆమోదాన్ని నియంత్రించడం**.

ఈ నమూనాలు నమ్మకదయైన, ఉత్పత్తి-సిద్ధ ఏజెంట్లను తయారు చేయడానికి పునాది చేర్చుతాయి, అవి బాహ్య వ్యవస్థలతో శ్రద్ధగా వ్యవహరిస్తాయి.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**అస్వీకరణ**:
ఈ పత్రం AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మేము ఖచ్చితత్వానికి ప్రయత్నిస్తున్నప్పటికీ, ఆటోమేటెడ్ అనువాదాలు తప్పులు లేదా అసమగ్రతలను కలిగి ఉండవచ్చు. దాని స్వదేశ భాషలో ఉన్న అసలు పత్రాన్ని అధికారం కలిగిన మూలంగా పరిగణించాలి. కీలకమైన సమాచారం కోసం, ప్రొఫెషనల్ మానవ అనువాదాన్ని సిఫారసు చేస్తాము. ఈ అనువాదం ఉపయోగం వల్ల కలిగే ఏవైనా అపార్థాలు లేదా తప్పుదారులు కోసం మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
